# ER triage acuity classification model
CSCI5622 Project \
Name: Sabrina Touch

### 1. Install packages, load data

In [64]:
#%pip install pandas
#%pip install numpy
#%pip install scikit-learn==1.8
#%pip install matplotlib
#%pip install jinja2

In [12]:
import numpy as np
import pandas as pd

from sklearn.metrics import recall_score, confusion_matrix, roc_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from scipy.stats import norm

# Load in data, set SEED
SEED = 42
data_original = pd.read_csv("./nhamcs_data_2018_22.csv")

print("Imported libraries, read in csv")

Imported libraries, read in csv


### Pre-split row-wise data adjustments on entire data

In [ ]:
target_col = 'target_triage_acuity'

data_full = data_original.copy()

# Only information available at registration may be a feature.

# Measured BY the triage nurse. Do not keep.
TRIAGE_MEASURED = ['temp', 'heart_rate', 'resp_rate', 'sys_bp', 'dias_bp', 'spo2', 'pain_score', 'vitals_during_visit']

# Determined AFTER triage -> leakage under any framing
POST_TRIAGE = ['wait_time_minutes', 'intervention_iv_fluids']

data_full = data_full.drop(columns=TRIAGE_MEASURED + POST_TRIAGE)

# Evaluate walk-in visits only. (ems_arrival = 'No')
n_before = len(data_full)
data_full = data_full[data_full['ems_arrival'] == 'No'].copy()
data_full = data_full.drop(columns=['ems_arrival'])      # constant now, so not a useful feature
print(f"Walk-in visits: {len(data_full):,} of {n_before:,} ({len(data_full)/n_before:.1%})")

# Safety check: none of the excluded columns survive
assert not set(TRIAGE_MEASURED + POST_TRIAGE) & set(data_full.columns)
print(f"Dropped {len(TRIAGE_MEASURED + POST_TRIAGE)} columns; {data_full.shape[1]} remain.")
print(f"{data_full.shape[0]}")


Walk-in visits: 46,588 of 58,124 (80.2%)
Dropped 10 columns; 38 remain.
46588


### Temporal split: train / validation / test by year
- **Train (2018–2020):** fit the model and tune hyperparameters (stratified 5-fold CV)
- **Validation (2021):** choose the decision threshold for the 15% FPR budget
- **Test (2022):** final evaluation, used once

Every split predicts a later period from an earlier one, matching deployment.

In [7]:
# Temporal split: train on the past, set the threshold on the next year, test on the most recent year
SPLIT_YEARS = {
    'train': [2018, 2019, 2020, 2021],   # fit the model and tune hyperparameters
    'test':  [2022],               # final evaluation, used once
}

def make_split(df, years):
    part = df[df['year'].isin(years)]
    X = part.drop(columns=[target_col, 'year'])      # year is not a model input
    y_esi = part[target_col]                          # full ESI 1-5 (for stratified CV and slice analysis)
    y_binary = y_esi.isin([1, 2]).astype(int)         # 1 = critical (ESI 1-2), 0 = non-critical (ESI 3-5)
    return X, y_binary, y_esi

X_train, y_train_binary, y_train = make_split(data_full, SPLIT_YEARS['train'])
X_test,  y_test_binary,  y_test  = make_split(data_full, SPLIT_YEARS['test'])

# Every row lands in exactly one set
assert len(X_train) + len(X_test) == len(data_full), "Rows lost or duplicated in split"

# Summary: size and share of critical patients per set
display(pd.DataFrame({
    'Years':          [', '.join(map(str, y)) for y in SPLIT_YEARS.values()],
    'Rows':           [len(X_train), len(X_test)],
    'Critical (ESI 1-2)': [y_train_binary.sum(), y_test_binary.sum()],
    'Critical share': [y_train_binary.mean(), y_test_binary.mean()],
}, index=list(SPLIT_YEARS)).style.format({'Rows': '{:,}', 'Critical share': '{:.1%}'}))

,Years,Rows,Critical (ESI 1-2),Critical share
train,"2018, 2019, 2020, 2021","38,460",4719,12.3%
test,2022,"8,128",1076,13.2%


### Confidence Interval Function

In [ ]:
# Confidence interval helper function
def wilson_ci(successes, n, conf=0.95):
    """Wilson score interval for a proportion successes/n."""
    z = norm.ppf(1 - (1 - conf) / 2)          # 1.96 for 95%
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - half, center + half

### Set, calculate baseline

In [ ]:
# Baseline rule: "If age >= 65, predict critical" (critical = ESI 1-2), evaluated on the 2022 test set
age_predictions = (X_test['age'] >= 65).astype(int)
age_recall = recall_score(y_test_binary, age_predictions)

tn, fp, fn, tp = confusion_matrix(y_test_binary, age_predictions).ravel()
age_fpr = fp / (fp + tn)
print(f"Age >= 65 baseline (2022 test): recall = {age_recall:.3f}, FPR = {age_fpr:.3f}")

age_recall_ci = wilson_ci(tp, tp + fn)   # recall: caught critical / all critical
age_fpr_ci    = wilson_ci(fp, fp + tn)   # FPR: rushed non-critical / all non-critical

print(f"Recall = {age_recall:.3f}  (95% CI {age_recall_ci[0]:.3f}-{age_recall_ci[1]:.3f}, n = {tp + fn})")
print(f"FPR    = {age_fpr:.3f}  (95% CI {age_fpr_ci[0]:.3f}-{age_fpr_ci[1]:.3f}, n = {fp + tn})")


Age >= 65 baseline (2022 test): recall = 0.263, FPR = 0.147
Recall = 0.263  (95% CI 0.238-0.290, n = 1076)
FPR    = 0.147  (95% CI 0.139-0.156, n = 7052)


### 5. Logistic Regression